# Credit card transactions generator

This notebook simulates a card-processing application that sends each transaction to a
Kinesis Data Streams stream, where a consumer (a Lambda, or the model from notebook 02
behind an endpoint) scores it for fraud.

The transactions are synthetic, generated by `frauddata.py` in the shape of the Kaggle
credit-card fraud dataset: `Time`, anonymised components `V1`..`V28`, `Amount`, and the
`Class` label. The notebook runs from a fresh clone, with no download.

**Code Cell 1**: a Kinesis client. On AWS this is `boto3.client("kinesis")`; oblako's points at its local Kinesis.

In [1]:
import csv
import tempfile
from pathlib import Path

import frauddata
from oblako.services import KinesisService

STREAM_NAME = "TransactionsStream"
kinesis_client = KinesisService().get_client()

**Code Cell 2**: create the stream, as the AWS solution's CloudFormation stack does, and wait until it is active.

In [2]:
if STREAM_NAME not in kinesis_client.list_streams()["StreamNames"]:
    kinesis_client.create_stream(StreamName=STREAM_NAME, ShardCount=1)
kinesis_client.get_waiter("stream_exists").wait(StreamName=STREAM_NAME)
print(
    kinesis_client.describe_stream_summary(StreamName=STREAM_NAME)[
        "StreamDescriptionSummary"
    ]["StreamStatus"]
)

ACTIVE


**Code Cell 3**: send transactions, one record each in CSV form, without the `Class` label (the consumer doesn't know the answer).

In [3]:
csv_path = Path(tempfile.mkdtemp()) / "transactions.csv"
frauddata.write_training_csv(str(csv_path), n=20, seed=42)

with open(csv_path, newline="") as fh:
    for n, row in enumerate(csv.DictReader(fh), start=1):
        row.pop("Class")
        record = ",".join(row.values())
        kinesis_client.put_record(
            StreamName=STREAM_NAME, Data=record, PartitionKey="partitionkey"
        )
print(f"sent {n} transactions; the last one:")
print(record[:120], "...")

sent 20 transactions; the last one:
46812,0.301948,0.375457,-0.51032,0.421576,1.737697,-1.1078,1.737426,-0.242803,-0.766627,2.427221,0.155022,1.370403,0.817 ...


**Code Cell 4**: read the stream from the start, as a consumer would.

In [4]:
shard = kinesis_client.list_shards(StreamName=STREAM_NAME)["Shards"][0]["ShardId"]
iterator = kinesis_client.get_shard_iterator(
    StreamName=STREAM_NAME, ShardId=shard, ShardIteratorType="TRIM_HORIZON"
)["ShardIterator"]
records = kinesis_client.get_records(ShardIterator=iterator, Limit=100)["Records"]
print(f"{len(records)} records on the stream")
first = records[0]["Data"].decode().split(",")
print(f"first: Time={first[0]}, Amount={first[-1]}")

20 records on the stream
first: Time=126092, Amount=51.36


**Code Cell 5**: clean up.

In [5]:
kinesis_client.delete_stream(StreamName=STREAM_NAME)
print("deleted", STREAM_NAME)

deleted TransactionsStream
